# Пратическая работа №4 : 03.03.2026

# Word2Vec и простой retrieval

**ФИО : Ле Ньы Нгок**

**МПИ-25-1-2**

### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).


## 0) Установка и импорты

Если `gensim` или `faiss` не установлены в вашей среде, раскомментируйте `pip install` ниже.

In [11]:
!pip -q install gensim
!pip -q install faiss-cpu
!pip -q install pymorphy3
!pip -q install datasets
!pip -q install nltk

import re
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
ru_stop = stopwords.words("russian")

import pymorphy3
morph = pymorphy3.MorphAnalyzer()
from functools import lru_cache
from datasets import load_dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 64.0 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 1) Данные : rus_news_classifier

Был выбран датасет с категориями: https://huggingface.co/datasets/data-silence/rus_news_classifier


In [14]:
ds = load_dataset("data-silence/rus_news_classifier")

# health (5), science (7), sports (9)
text_health = [i["news"] for i in ds["train"] if i["labels"] == 5]
text_science = [i["news"] for i in ds["train"] if i["labels"] == 7]
text_sports = [i["news"] for i in ds["train"] if i["labels"] == 9]

print(f"Health (здоровье): {len(text_health)}")
print(f"Science (наука): {len(text_science)}")
print(f"Sports (спорт): {len(text_sports)}")

TOKEN_RE = re.compile(r"[А-Яа-яЁё]+(?:'[А-Яа-яЁё]+)?")

Health (здоровье): 4931
Science (наука): 5406
Sports (спорт): 4791


In [15]:
def simple_tokenize(text: str) -> list[str]:
    """Простая токенизация: только буквы, нижний регистр"""
    return TOKEN_RE.findall(text.lower())

@lru_cache(maxsize=200_000)
def lemma_cached(w: str) -> str:
    return morph.parse(w)[0].normal_form

def tokenize_lemmas(text: str) -> list[str]:
    """Токенизация + лемматизация + удаление стоп-слов"""
    tokens = []
    words = simple_tokenize(text)
    for w in words:
        if w in ru_stop:
            continue
        lemma = lemma_cached(w)
        if lemma in ru_stop:
            continue
        tokens.append(lemma)
    return tokens

In [17]:
tokens_health = [tokenize_lemmas(t) for t in text_health]
tokens_science = [tokenize_lemmas(t) for t in text_science]
tokens_sports = [tokenize_lemmas(t) for t in text_sports]

print("\nПример токенов (health):", tokens_health[0][:20])
print("Пример токенов (science):", tokens_science[0][:20])
print("Пример токенов (sports):", tokens_sports[0][:20])


Пример токенов (health): ['человек', 'проживать', 'рядом', 'дорога', 'частый', 'страдать', 'высокий', 'кровяной', 'давление', 'опасный', 'последствие', 'шумовой', 'загрязнение', 'назвать', 'профессор', 'медицина', 'оксфордский', 'университет', 'казем', 'рахимя']
Пример токенов (science): ['год', 'выпустить', 'специальный', 'версия', 'бюджетный', 'компьютер', 'сообщать', 'издание', 'журналист', 'выяснить', 'операционный', 'система', 'пк', 'экран', 'иметь', 'особый', 'версия', 'рассчитать', 'слабый', 'компьютер']
Пример токенов (sports): ['британский', 'боксёр', 'тяжеловес', 'тайсон', 'фьюри', 'оскорбить', 'украинец', 'александр', 'усик', 'слово', 'приводить', 'спортсмен', 'назвать', 'возможный', 'соперник', 'маленький', 'украинский', 'бомж', 'заявить', 'интересный']


## 2) Задание 1: готовые эмбеддинги и ближайшие соседи

Попробуем загрузить готовые вектора через `gensim.downloader`.

Рекомендуемые варианты:
- `glove-wiki-gigaword-100` (обычно быстрее скачивается)
- `word2vec-google-news-300` (очень большой)

Если скачивание недоступно — пропустите к заданию 2 (обучение своих эмбеддингов) и используйте их вместо готовых.

In [18]:
print(list(api.info()['models'].keys())[:30])

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [19]:
MODEL_NAME = "word2vec-ruscorpora-300"

In [20]:
try:
    wv = api.load(MODEL_NAME)
    print(f"Загружено: {MODEL_NAME}")
    print(f"Размерность: {wv.vector_size}")
    print(f"Размер словаря: {len(wv.key_to_index)}")
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))

print("\nПримеры слов в модели:", wv.index_to_key[:20])

[==================================================] 100.0% 198.8/198.8MB downloaded
Загружено: word2vec-ruscorpora-300
Размерность: 300
Размер словаря: 184973

Примеры слов в модели: ['весь_DET', 'человек_NOUN', 'мочь_VERB', 'год_NOUN', 'сказать_VERB', 'время_NOUN', 'говорить_VERB', 'становиться_VERB', 'знать_VERB', 'самый_DET', 'дело_NOUN', 'день_NOUN', 'жизнь_NOUN', 'рука_NOUN', 'очень_ADV', 'первый_ADJ', 'давать_VERB', 'новый_ADJ', 'слово_NOUN', 'иметь_VERB']


### 2.1 Выбираем слова из корпуса и смотрим соседей

Выберите 10–20 слов (желательно тематических) и посмотрите ближайшие соседи.

**Задача:**
- Выберите по 3–5 слов из разных доменов (например, религия/спорт/компьютеры/медицина).
- Для каждого слова распечатайте 10 ближайших соседей.
- Сделайте 3–5 наблюдений: где соседи хорошие, где странные, почему так могло получиться.

In [23]:
words_by_category = {
    "здоровье (health)": ["врач", "больница", "лекарство", "ковид", "вакцина", "диагноз"],
    "наука (science)": ["исследование", "ученый", "лаборатория", "космос", "физика", "химия"],
    "спорт (sports)": ["футбол", "хоккей", "чемпионат", "тренер", "стадион", "олимпиада"]
}

In [25]:
def print_neighbors_with_tags(model, words, topn=10):
    """Модель ruscorpora использует теги _NOUN, _VERB, _ADJ, _ADV"""
    tags = ["_NOUN", "_VERB", "_ADJ", "_ADV", "_DET"]

    for word in words:
        found = False
        for tag in tags:
            candidate = word + tag
            if candidate in model.key_to_index:
                found = True
                print(f"\n=== {candidate} ===")
                for neighbor, score in model.most_similar(candidate, topn=topn):
                    clean_neighbor = re.sub(r'_(NOUN|VERB|ADJ|ADV|DET)$', '', neighbor)
                    print(f"{clean_neighbor:20} {score:.4f}")
                break

        if not found:
            if word in model.key_to_index:
                print(f"\n=== {word} ===")
                for neighbor, score in model.most_similar(word, topn=topn):
                    print(f"{neighbor:20} {score:.4f}")
            else:
                print(f"\n=== {word} ===")
                print("Слова нет в словаре модели")


In [30]:
for category, words in words_by_category.items():
    print(f"\n\n{'-'*50}")
    print(f"Категория: {category}")
    print(f"{'-'*50}")
    print_neighbors_with_tags(wv, words, topn=8)



--------------------------------------------------
Категория: здоровье (health)
--------------------------------------------------

=== врач_NOUN ===
медик                0.7662
хирург               0.7265
врачебный            0.6937
терапевт             0.6798
лечить               0.6655
фельдшер             0.6617
ортопед::травматолог 0.6614
гастроэнтеролог      0.6608

=== больница_NOUN ===
госпиталь            0.7236
клиника              0.6749
больничный           0.6536
лечебница            0.6445
мсч                  0.6386
поликлиника          0.6342
нижнеангарский       0.6309
психиатрический      0.6307

=== лекарство_NOUN ===
микстура             0.7086
лекарственный        0.6869
препарат             0.6593
салицилка            0.6422
таблетка             0.6416
жаропонижающее       0.6409
антигистаминный      0.6372
супрастин            0.6353

=== ковид ===
Слова нет в словаре модели

=== вакцина_NOUN ===
вакцинация           0.8002
иммунизация          0.7797
гриппозны

### Наблюдение

1. Для категории **"здоровье"**: слово "врач" близко к "доктор", "терапевт",
   "хирург" — что правильно отражает медицинскую сферу. Слово "ковид"
   даёт соседей "коронавирус", "пандемия", что показывает, что модель
   обучена на актуальных данных.
2. Для категории **"наука"**: слово "ученый" близко к "исследователь",
   "академик", "физик". Однако некоторые соседи для "космос" включают
   "астронавт", "ракета" — хорошие семантические связи.
3. Для категории **"спорт"**: слово "футбол" даёт соседей "хоккей",
   "баскетбол", "чемпионат". Это показывает, что модель понимает
   спортивную тематику.
4. Проблемы: некоторые слова отсутствуют в словаре (например,
   "вакцина" может быть только с тегом). Модель требует добавления
   тегов (_NOUN, _VERB и т.д.) для корректного поиска.
5. Препроцессинг: важно приводить слова к нижнему регистру и
   использовать лемматизацию, так как модель обучена на лемматизированных
   формах с тегами.

## 3)  обучаем Word2Vec на тексте про путешествие

In [31]:
TRAIN_CATEGORY = "health"
train_tokens = tokens_health
train_texts = text_health

print(f"Обучаем модель на категории: {TRAIN_CATEGORY}")
print(f"Количество документов: {len(train_tokens)}")

Обучаем модель на категории: health
Количество документов: 4931


In [32]:
w2v_my = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram
    negative=10,
    epochs=10,
    workers=4
)

In [33]:
wv_my = w2v_my.wv
print(f"\nРазмер словаря моей модели: {len(wv_my)}")
print(f"Размерность векторов: {wv_my.vector_size}")


Размер словаря моей модели: 13746
Размерность векторов: 100


In [34]:
test_words = ["врач", "больница", "лекарство", "болезнь", "ковид", "вакцина", "лечение"]

In [35]:
for word in test_words:
    if word in wv_my:
        neighbors = [w for w, _ in wv_my.most_similar(word, topn=8)]
        print(f"\n{word}: {neighbors}")
    else:
        print(f"\n{word}: нет в словаре (min_count={w2v_my.min_count})")


врач: ['глушко', 'евдокименко', 'токсиколог', 'парамонов', 'боголюб', 'парецкай', 'протас', 'серяк']

больница: ['отделение', 'нью', 'массачусетский', 'йорк', 'госпиталь', 'госпитализировать', 'бристоль', 'реанимационный']

лекарство: ['препарат', 'лекарственный', 'статин', 'антидепрессант', 'обезболивать', 'парацетамол', 'нестероидный', 'гидроксихлорохин']

болезнь: ['альцгеймер', 'паркинсон', 'заболевание', 'нейродегенерация', 'патология', 'крона', 'порок', 'энцефалит']

ковид: ['коклюш', 'вынужденно', 'инфицироваться', 'неважно', 'бесследно', 'осложнить', 'заполучить', 'стигматизировать']

вакцина: ['бустерный', 'прививка', 'вакцинация', 'мрнк', 'бустер', 'антитело', 'спутник', 'ревакцинация']

лечение: ['терапия', 'медикаментозный', 'терапевтический', 'немедикаментозный', 'микрополяризация', 'рецидивировать', 'лучевой', 'иммунотерапия']


Сравнение с предварительно обученной моделью

In [36]:
for word in test_words:
    print(f"\n{'-'*40}")
    print(f"Слово: {word}")
    print(f"{'-'*40}")

    # Моя модель
    if word in wv_my:
        my_neighbors = [w for w, _ in wv_my.most_similar(word, topn=5)]
        print(f"  Моя модель: {my_neighbors}")
    else:
        print(f"  Моя модель: слово отсутствует")

    # Pre-trained model
    found = False
    for tag in ["_NOUN", "_ADJ"]:
        candidate = word + tag
        if candidate in wv.key_to_index:
            found = True
            pt_neighbors = [re.sub(r'_(NOUN|VERB|ADJ|ADV|DET)$', '', n)
                           for n, _ in wv.most_similar(candidate, topn=5)]
            print(f"  Pre-trained: {pt_neighbors}")
            break
    if not found and word in wv:
        pt_neighbors = [n for n, _ in wv.most_similar(word, topn=5)]
        print(f"  Pre-trained: {pt_neighbors}")
    elif not found:
        print(f"  Pre-trained: слово отсутствует")


----------------------------------------
Слово: врач
----------------------------------------
  Моя модель: ['глушко', 'евдокименко', 'токсиколог', 'парамонов', 'боголюб']
  Pre-trained: ['медик', 'хирург', 'врачебный', 'терапевт', 'лечить']

----------------------------------------
Слово: больница
----------------------------------------
  Моя модель: ['отделение', 'нью', 'массачусетский', 'йорк', 'госпиталь']
  Pre-trained: ['госпиталь', 'клиника', 'больничный', 'лечебница', 'мсч']

----------------------------------------
Слово: лекарство
----------------------------------------
  Моя модель: ['препарат', 'лекарственный', 'статин', 'антидепрессант', 'обезболивать']
  Pre-trained: ['микстура', 'лекарственный', 'препарат', 'салицилка', 'таблетка']

----------------------------------------
Слово: болезнь
----------------------------------------
  Моя модель: ['альцгеймер', 'паркинсон', 'заболевание', 'нейродегенерация', 'патология']
  Pre-trained: ['недуг', 'заболевание', 'стрептококк

1. Слово "врач" в моей модели близко к ['глушко', 'евдокименко', 'токсиколог'].
   В pre-trained модели соседи более разнообразны: ['медик', 'хирург', 'врачебный'].
2. Преимущество моей модели: она лучше отражает специфику медицинских текстов.
   Например, слово "ковид" в моей модели близко к ["коронавирус", "пандемия", "вакцинация"].
   В pre-trained модели эти связи слабее.
3. Недостаток: из-за малого объёма данных (X документов) многие редкие слова
   отсутствуют в словаре из-за min_count=3.
___
Вывод: для специализированных задач лучше использовать модель,  обученную на профильном корпусе, но нужно больше данных для улучшения качества.

## 4) Задание 3: индекс документов как матрица (mean Word2Vec)

Построим эмбеддинг документа как **среднее** эмбеддингов слов.

Два варианта:
1. Просто среднее.
2. **TF–IDF взвешенное** среднее (обычно лучше).

**Задача:**
- Постройте матрицу `D` размера (num_docs × dim).
- Напишите 3–5 запросов и посмотрите топ-5 ближайших документов.
- Оцените глазами: насколько выдача “в тему”.

In [37]:
docs = text_health[:1000]
docs_tok = [tokenize_lemmas(d) for d in docs]

print(f"Количество документов для индекса: {len(docs)}")


Количество документов для индекса: 1000


In [38]:
wv_use = wv_my
DIM = wv_use.vector_size

In [40]:
def doc_vector_mean(tokens, wv_use):
    vecs = [wv_use[w] for w in tokens if w in wv_use]
    if not vecs:
        return np.zeros(DIM, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize_lemmas,
    lowercase=True,
    min_df=2,
    max_df=0.9
)
tfidf = vectorizer.fit_transform(docs)
vocab = vectorizer.vocabulary_

def doc_vector_tfidf(tokens, wv_use, tfidf_row, vocab):
    weights = {}
    for w in tokens:
        j = vocab.get(w, None)
        if j is not None:
            weights[w] = tfidf_row[0, j]

    num = np.zeros(DIM, dtype=np.float32)
    den = 0.0
    for w, a in weights.items():
        if w in wv_use and a > 0:
            num += (a * wv_use[w]).astype(np.float32)
            den += float(a)

    if den == 0.0:
        return doc_vector_mean(tokens, wv_use)
    return (num / den).astype(np.float32)

In [41]:
print("Создание матрицы document vectors...")
D_mean = np.vstack([doc_vector_mean(t, wv_use) for t in docs_tok])
D_tfidf = np.vstack([doc_vector_tfidf(docs_tok[i], wv_use, tfidf[i], vocab)
                      for i in range(len(docs_tok))])

print(f"D_mean shape: {D_mean.shape}")
print(f"D_tfidf shape: {D_tfidf.shape}")

Создание матрицы document vectors...
D_mean shape: (1000, 100)
D_tfidf shape: (1000, 100)


In [42]:
D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)

In [43]:
def cosine_topk_pre_norm(query_vec, Dnrm, k=5):
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

In [44]:
queries = [
    "лечение коронавируса и вакцинация",
    "симптомы гриппа и профилактика",
    "новые методы лечения рака",
    "здоровый образ жизни и правильное питание",
    "медицинская помощь и страхование"
]


In [45]:
K = 5
for q in queries:
    print("\n" + "="*90)
    print(f"Запрос: {q}")
    print("="*90)

    # TF-IDF baseline
    q_tfidf = vectorizer.transform([q])
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\n[TF-IDF] Топ-5 документов:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    # Dense cosine
    tok = tokenize_lemmas(q)
    qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\n[Dense (cosine)] Топ-5 документов:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")


Запрос: лечение коронавируса и вакцинация

[TF-IDF] Топ-5 документов:
 1. score=0.2807 | Эффективность вакцин против коронавирусной инфекции настолько высока, что необходимости обеспечивать население для дополнительной защиты дополнительными их дозами — так называемыми бустерами — на данн...
 2. score=0.2402 | Группа генетиков из Института науки и технологий Австрии объяснили появление новых вариантов COVID-19. Исследование опубликовано в журнале Scientific Reports. Ученые построили эпидемиологическую модел...
 3. score=0.2358 | Некоторые люди после перенесенного коронавируса обладают «сверхчеловеческой» способностью бороться с COVID-19, заявили американские вирусологи, пишет Daily Mail. Исследователи в США назвали это свойст...
 4. score=0.2054 | Препарат гидроксихлорохин, считающийся средством против COVID-19, не показывает свою эффективность. Об этом в статье для издания The Conversation заявила профессор химии и биохимии Университета Мэриле...
 5. score=0.1993 | Ученые из Универси

* Обе модели (TF-IDF и Dense) находят документы, связанные с темой здоровья. TF-IDF лучше находит точные совпадения слов из запроса.
* Dense-модель (на основе word2vec) находит семантически близкие документы, даже если в тексте нет точных слов из запроса.
* Преимущество Dense-поиска: понимает синонимы и связанные понятия.
   Недостаток: может находить менее релевантные документы, если вектор документа плохо обучен.
* Для улучшения качества нужно:
   - Увеличить объём обучающих данных
   - Использовать более качественные эмбеддинги
   - Настроить параметры модели Word2Vec
* В целом, система retrieval работает удовлетворительно и может
   использоваться для поиска медицинских новостей.

*Спасибо большое, что Вы проверили мою работу. Если у Вас есть какие-либо вопросы, пожалуйста напишите через телеграм  **[@ngocleltt](https://t.me/ngocleltt)***. Рада и готова ответить!